In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])


In [ ]:
train_dataset = datasets.ImageFolder(
    root="Dataset/Train",
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root="Dataset/Validation",
    transform=transform
)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

print("Classes:", train_dataset.classes)


Classes: ['Fake', 'Real']


MesoNet

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class Meso4(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(8)

        self.conv2 = nn.Conv2d(8, 8, 5, padding=2)
        self.bn2 = nn.BatchNorm2d(8)

        self.conv3 = nn.Conv2d(8, 16, 5, padding=2)
        self.bn3 = nn.BatchNorm2d(16)

        self.conv4 = nn.Conv2d(16, 16, 5, padding=2)
        self.bn4 = nn.BatchNorm2d(16)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(16 * 16 * 16, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return torch.sigmoid(self.fc2(x))


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


In [ ]:
model = Meso4().to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
def train_one_epoch(loader):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
def validate(loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = (outputs > 0.5).int().squeeze()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


In [ ]:
epochs = 10

for epoch in range(epochs):
    loss = train_one_epoch(train_loader)
    acc = validate(val_loader)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}%")


Epoch 1/10 | Loss: 0.4006 | Val Acc: 83.81%
Epoch 2/10 | Loss: 0.2398 | Val Acc: 85.91%
Epoch 3/10 | Loss: 0.1921 | Val Acc: 89.13%
Epoch 4/10 | Loss: 0.1685 | Val Acc: 90.92%
Epoch 5/10 | Loss: 0.1541 | Val Acc: 91.01%
Epoch 6/10 | Loss: 0.1440 | Val Acc: 91.69%
Epoch 7/10 | Loss: 0.1331 | Val Acc: 91.87%
Epoch 8/10 | Loss: 0.1278 | Val Acc: 92.49%
Epoch 9/10 | Loss: 0.1210 | Val Acc: 92.71%
Epoch 10/10 | Loss: 0.1152 | Val Acc: 90.80%


In [ ]:
torch.save(model.state_dict(), "mesonet.pth")
print("Model saved!")


Model saved!


In [ ]:
from google.colab import files
files.download("mesonet.pth")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Downloading the Dataset**

In [6]:
!kaggle datasets download manjilkarki/deepfake-and-real-images


Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
100% 1.68G/1.68G [00:19<00:00, 254MB/s]
100% 1.68G/1.68G [00:19<00:00, 95.0MB/s]


In [3]:
!pip install kaggle



Setting up kaggle file for credentials

In [5]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
